# LoRA + QLoRA Fine-tuning Notebook (Llama 3.2 1B Instruct)

This notebook is split into **independent sections** so you can run tasks separately:
1. Baseline (no fine-tuning) + RAG evaluation
2. QLoRA fine-tuning + validation/testing + RAG evaluation
3. LoRA fine-tuning + validation/testing + RAG evaluation

It uses sample instruction data from the internet via Hugging Face Datasets.


## 0) Environment setup (run once)


In [ ]:
# If needed, uncomment:
# !pip install -q -U transformers datasets peft trl accelerate bitsandbytes sentence-transformers faiss-cpu evaluate

import os
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


## 1) Shared configuration and utilities


In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    BitsAndBytesConfig,
    pipeline,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer
from sentence_transformers import SentenceTransformer
import faiss
import evaluate

BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
DATASET_NAME = "yahma/alpaca-cleaned"
TEXT_COL = "text"
MAX_TRAIN = 3000
MAX_VAL = 300
MAX_TEST = 300
OUTPUT_ROOT = "artifacts"
os.makedirs(OUTPUT_ROOT, exist_ok=True)

rouge = evaluate.load("rouge")


def format_alpaca(example):
    instruction = example.get("instruction", "")
    input_text = example.get("input", "")
    output = example.get("output", "")
    prompt = f"### Instruction:
{instruction}

"
    if input_text.strip():
        prompt += f"### Input:
{input_text}

"
    prompt += f"### Response:
{output}"
    return {TEXT_COL: prompt}


def load_data():
    ds = load_dataset(DATASET_NAME, split="train")
    ds = ds.map(format_alpaca, remove_columns=ds.column_names)
    ds = ds.shuffle(seed=SEED)

    train = ds.select(range(min(MAX_TRAIN, len(ds))))
    val = ds.select(range(min(MAX_TRAIN, len(ds)), min(MAX_TRAIN + MAX_VAL, len(ds))))
    test = ds.select(range(min(MAX_TRAIN + MAX_VAL, len(ds)), min(MAX_TRAIN + MAX_VAL + MAX_TEST, len(ds))))
    return train, val, test


def build_generator(model, tokenizer):
    return pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto" if torch.cuda.is_available() else None,
    )


def generate_answer(gen, prompt, max_new_tokens=128):
    out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
    return out[0]["generated_text"][len(prompt):].strip()


def evaluate_generation(gen, dataset, n=50):
    refs, preds = [], []
    for row in dataset.select(range(min(n, len(dataset)))):
        full = row[TEXT_COL]
        split_key = "### Response:
"
        p, ref = full.split(split_key, 1)
        prompt = p + split_key
        pred = generate_answer(gen, prompt)
        refs.append(ref.strip())
        preds.append(pred)
    scores = rouge.compute(predictions=preds, references=refs)
    return scores


## 2) Build a tiny RAG pipeline for benchmark comparisons

We use validation examples as the retrieval corpus and evaluate QA-style generation.


In [ ]:
def build_rag_index(corpus_texts):
    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embs = embedder.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return embedder, index


def rag_answer(gen, embedder, index, corpus_texts, question, top_k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)
    _, idx = index.search(q_emb, top_k)
    ctx = "

".join([corpus_texts[i] for i in idx[0]])
    prompt = f"Use the context to answer the question.\n\nContext:\n{ctx}\n\nQuestion:\n{question}\n\nAnswer:"
    return generate_answer(gen, prompt, max_new_tokens=96)


def evaluate_rag(gen, rag_components, dataset, n=30):
    embedder, index, corpus_texts = rag_components
    refs, preds = [], []
    for row in dataset.select(range(min(n, len(dataset)))):
        full = row[TEXT_COL]
        split_key = "### Response:
"
        p, ref = full.split(split_key, 1)
        question = p.replace("### Instruction:
", "").replace("### Input:
", "
").strip()
        pred = rag_answer(gen, embedder, index, corpus_texts, question)
        refs.append(ref.strip())
        preds.append(pred)
    return rouge.compute(predictions=preds, references=refs)


## 3) Load data + baseline model (before fine-tuning)


In [ ]:
train_ds, val_ds, test_ds = load_data()
print(train_ds, val_ds, test_ds)

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)

baseline_gen = build_generator(base_model, base_tokenizer)


### 3.1 Baseline testing + validation


In [ ]:
baseline_val_scores = evaluate_generation(baseline_gen, val_ds, n=40)
baseline_test_scores = evaluate_generation(baseline_gen, test_ds, n=40)
print("Baseline validation:", baseline_val_scores)
print("Baseline test:", baseline_test_scores)


### 3.2 Baseline RAG evaluation


In [ ]:
corpus_texts = [x[TEXT_COL] for x in val_ds.select(range(min(250, len(val_ds))))]
rag_embedder, rag_index = build_rag_index(corpus_texts)
rag_components = (rag_embedder, rag_index, corpus_texts)

baseline_rag_scores = evaluate_rag(baseline_gen, rag_components, test_ds, n=30)
print("Baseline RAG:", baseline_rag_scores)


---
## 4) Section A — QLoRA fine-tuning (independent)


In [ ]:
QLORA_OUT = os.path.join(OUTPUT_ROOT, "qlora")
os.makedirs(QLORA_OUT, exist_ok=True)

bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

qlora_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",
)
qlora_model.config.use_cache = False
qlora_model = prepare_model_for_kbit_training(qlora_model)

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
qlora_model = get_peft_model(qlora_model, qlora_config)
qlora_model.print_trainable_parameters()

qlora_args = TrainingArguments(
    output_dir=QLORA_OUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=torch.cuda.is_available(),
    bf16=False,
    report_to="none",
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    tokenizer=base_tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field=TEXT_COL,
    max_seq_length=512,
    args=qlora_args,
)

# Run independently when ready
# qlora_trainer.train()
# qlora_trainer.save_model(QLORA_OUT)


### 4.1 QLoRA validation/testing evaluation


In [ ]:
# Uncomment after training:
# qlora_infer_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto" if torch.cuda.is_available() else None)
# qlora_infer_model = PeftModel.from_pretrained(qlora_infer_model, QLORA_OUT)
# qlora_gen = build_generator(qlora_infer_model, base_tokenizer)
# qlora_val_scores = evaluate_generation(qlora_gen, val_ds, n=40)
# qlora_test_scores = evaluate_generation(qlora_gen, test_ds, n=40)
# print("QLoRA validation:", qlora_val_scores)
# print("QLoRA test:", qlora_test_scores)


### 4.2 QLoRA RAG evaluation


In [ ]:
# Uncomment after qlora_gen is created:
# qlora_rag_scores = evaluate_rag(qlora_gen, rag_components, test_ds, n=30)
# print("QLoRA RAG:", qlora_rag_scores)


---
## 5) Section B — LoRA fine-tuning (independent)


In [ ]:
LORA_OUT = os.path.join(OUTPUT_ROOT, "lora")
os.makedirs(LORA_OUT, exist_ok=True)

lora_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
lora_model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

lora_args = TrainingArguments(
    output_dir=LORA_OUT,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy="epoch",
    evaluation_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

lora_trainer = SFTTrainer(
    model=lora_model,
    tokenizer=base_tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field=TEXT_COL,
    max_seq_length=512,
    args=lora_args,
)

# Run independently when ready
# lora_trainer.train()
# lora_trainer.save_model(LORA_OUT)


### 5.1 LoRA validation/testing evaluation


In [ ]:
# Uncomment after training:
# lora_infer_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto" if torch.cuda.is_available() else None)
# lora_infer_model = PeftModel.from_pretrained(lora_infer_model, LORA_OUT)
# lora_gen = build_generator(lora_infer_model, base_tokenizer)
# lora_val_scores = evaluate_generation(lora_gen, val_ds, n=40)
# lora_test_scores = evaluate_generation(lora_gen, test_ds, n=40)
# print("LoRA validation:", lora_val_scores)
# print("LoRA test:", lora_test_scores)


### 5.2 LoRA RAG evaluation


In [ ]:
# Uncomment after lora_gen is created:
# lora_rag_scores = evaluate_rag(lora_gen, rag_components, test_ds, n=30)
# print("LoRA RAG:", lora_rag_scores)


## 6) Final comparison table (baseline vs QLoRA vs LoRA, with/without RAG)


In [ ]:
import pandas as pd

rows = [
    ["Baseline", baseline_val_scores.get("rougeL", None), baseline_test_scores.get("rougeL", None), baseline_rag_scores.get("rougeL", None)],
    ["QLoRA", locals().get("qlora_val_scores", {}).get("rougeL", None) if isinstance(locals().get("qlora_val_scores", {}), dict) else None,
              locals().get("qlora_test_scores", {}).get("rougeL", None) if isinstance(locals().get("qlora_test_scores", {}), dict) else None,
              locals().get("qlora_rag_scores", {}).get("rougeL", None) if isinstance(locals().get("qlora_rag_scores", {}), dict) else None],
    ["LoRA",  locals().get("lora_val_scores", {}).get("rougeL", None) if isinstance(locals().get("lora_val_scores", {}), dict) else None,
              locals().get("lora_test_scores", {}).get("rougeL", None) if isinstance(locals().get("lora_test_scores", {}), dict) else None,
              locals().get("lora_rag_scores", {}).get("rougeL", None) if isinstance(locals().get("lora_rag_scores", {}), dict) else None],
]

summary = pd.DataFrame(rows, columns=["Model", "Validation ROUGE-L", "Test ROUGE-L", "RAG Test ROUGE-L"])
summary
